In [ ]:
from __future__ import annotations

import concurrent.futures
import dataclasses
import itertools
from collections.abc import Callable, Sequence
import multiprocessing
from typing import Any, Final, Protocol, final, overload, override
import concurrent.futures

import numpy as np
import numpy.typing as npt

np.set_printoptions(threshold=10_000)
np.set_printoptions(linewidth=10_000)


# ==================================================================================================
# Controller
#
# V             - value         - O(T)      ∈ [0; +∞)
# D             - debt          - O(exp(T)) ∈ [0; +∞)
# d = ln(D + 1) - remapped debt - O(T)      ∈ [0; +∞)


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class GridParams:
    D_min: float
    D_max: float
    D_int: int

    T0: float
    T1: float

    CFL: float
    Δt_min: float
    Δt_max: float

    def to_segment_space(self, k: float) -> GridParams:
        return GridParams(
            D_min=self.D_min * k,
            D_max=self.D_max * k,
            D_int=self.D_int,
            T0=self.T0,
            T1=self.T1,
            CFL=self.CFL,
            Δt_min=self.Δt_min,
            Δt_max=self.Δt_max,
        )


@final
class ValueFunctionLayer:
    __slots__: Final = ("__x0", "__x1", "__xs", "__iΔx")

    def __init__(self, x0: float, x1: float, n_int: int) -> None:
        assert x1 > x0
        self.__x0: Final = x0
        self.__x1: Final = x1
        self.__xs: Final = np.zeros(n_int + 2, dtype=np.float32)
        self.__iΔx: Final = (n_int + 1) / (x1 - x0)

    @property
    def x0(self) -> float:
        return self.__x0

    @property
    def x1(self) -> float:
        return self.__x1

    @property
    def iΔx(self) -> float:
        return self.__iΔx

    @property
    def w(self) -> npt.NDArray[np.float32]:
        return self.__xs

    def lookup(self, x_vals: npt.NDArray[np.float32]) -> npt.NDArray[np.float32]:
        i: Final = (x_vals - self.__x0) * self.__iΔx

        # Clamp indices for interpolation/extrapolation
        iM = np.clip(np.floor(i).astype(int), 0, len(self.__xs) - 2)
        iP = np.minimum(iM + 1, len(self.__xs) - 1)

        # Linear interpolation
        α = i - iM
        result = (1 - α) * self.__xs[iM] + α * self.__xs[iP]

        # Handle out-of-bounds (below x0)
        result = np.where(i < 0, -np.inf, result)

        return result


class Model(Protocol):
    def dotV(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]: ...
    def dotD(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]: ...


@final
class DSpace:
    def __init__(self, D_min: float, D_max: float, n_int: int) -> None:
        self.__d0: Final = self.D_to_d(D_min)
        self.__d1: Final = self.D_to_d(D_max)
        self.__Δd: Final = (self.__d1 - self.__d0) / (n_int + 1)
        self.__n_int: Final = n_int

    @property
    def d0(self) -> float:
        return self.__d0

    @property
    def d1(self) -> float:
        return self.__d1

    @property
    def Δd(self) -> float:
        return self.__Δd

    def D_to_k(self, D: float, *, clamp: bool) -> int:
        return self.d_to_k(self.D_to_d(D), clamp=clamp)

    def d_to_k(self, d: float, *, clamp: bool) -> int:
        k: Final = int(d / self.__Δd)
        if clamp:
            return min(max(0, k), self.__n_int + 1)
        else:
            return k

    def k_to_D(self, k: int) -> float:
        return self.d_to_D(self.k_to_d(k))

    def k_to_d(self, k: int) -> float:
        return self.d0 + k * self.Δd

    def dotD_to_dotd(self, dotD: npt.NDArray[np.float32], D: npt.NDArray[np.float32]) -> npt.NDArray[np.float32]:
        return dotD / (1 + D)

    @overload
    def D_to_d(self, D: float) -> float: ...

    @overload
    def D_to_d(self, D: npt.NDArray[np.float32]) -> npt.NDArray[np.float32]: ...

    def D_to_d(self, D: Any) -> Any:
        return np.log(1 + D)

    @overload
    def d_to_D(self, d: float) -> float: ...

    @overload
    def d_to_D(self, d: npt.NDArray[np.float32]) -> npt.NDArray[np.float32]: ...

    def d_to_D(self, d: Any) -> Any:
        return np.exp(d) - 1


@final
class AdaptiveStep:
    def __init__(self, cfl: float, Δt_min: float, Δt_max: float) -> None:
        self.__max_dot: float = -np.inf
        self.__Δt_min: Final = Δt_min
        self.__Δt_max: Final = Δt_max
        self.__cfl: Final = cfl
        self.__stable = True

    @property
    def stable(self) -> bool:
        return self.__stable

    def reset(self) -> None:
        self.__max_dot = -np.inf

    def consume_dotx(self, dotx: float) -> None:
        self.__max_dot = max(self.__max_dot, dotx)

    def compute_Δt(self, Δx: float) -> float:
        Δt: Final = self.__cfl * Δx / self.__max_dot
        if Δt >= self.__Δt_min:
            return min(Δt, self.__Δt_max)
        self.__stable = False
        return self.__Δt_min


@final
class Controller:
    __slots__: Final = ("__gp", "__model", "__ts", "__bs", "__stable", "__dsp")

    def __init__(self, model: Model, gp: GridParams) -> None:
        ts: Final = list[float]()
        bs: Final = list[npt.NDArray[np.bool]]()

        dsp: Final = DSpace(gp.D_min, gp.D_max, gp.D_int)
        Δt_step: Final = AdaptiveStep(gp.CFL, gp.Δt_min, gp.Δt_max)

        # NOTE: W(:, T) = 0
        curr_ws = ValueFunctionLayer(dsp.d0, dsp.d1, gp.D_int)
        next_ws = ValueFunctionLayer(dsp.d0, dsp.d1, gp.D_int)

        # Pre-compute the spatial grid
        kds: Final = np.arange(0, gp.D_int + 2, dtype=np.int32)
        d_grid: Final = dsp.d0 + kds * dsp.Δd
        D_grid: Final = dsp.d_to_D(d_grid)

        def compute_dotd(D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
            return dsp.dotD_to_dotd(model.dotD(D, b), D)

        def compute_dotv(D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
            return model.dotV(D, b)

        def compute_w_plus(
            d: npt.NDArray[np.float32], D: npt.NDArray[np.float32], b: bool, Δt: float
        ) -> npt.NDArray[np.float32]:
            dotd: Final = compute_dotd(D, b)
            dotV: Final = compute_dotv(D, b)
            Δt_step.consume_dotx(float(np.max(np.abs(dotd))))

            next_d: Final = d + dotd * np.float32(Δt)
            next_w: Final = next_ws.lookup(next_d) + dotV * np.float32(Δt)  # type: ignore
            return next_w  # type: ignore

        # Compute first adaptive time step
        dotd_0 = compute_dotd(D_grid, False)
        dotd_1 = compute_dotd(D_grid, True)

        max_dotd = max(np.max(np.abs(dotd_0)), np.max(np.abs(dotd_1)))
        Δt_step.consume_dotx(float(max_dotd))
        Δt = Δt_step.compute_Δt(dsp.Δd)

        ts.append(gp.T1)
        bs.append(np.ones_like(curr_ws.w, dtype=np.bool))

        t = gp.T1 - Δt

        next_round = True
        while next_round:
            # compute a current layer time
            t -= Δt

            next_round = t > gp.T0
            if not next_round:
                t = gp.T0

            # reset layer params
            Δt_step.reset()

            # layer-local params
            curr_bs = np.zeros_like(curr_ws.w, dtype=np.bool)

            # Determine optimal control (kd=0 -> b=True from stability purposes)
            ws_0 = compute_w_plus(d_grid[1:], D_grid[1:], False, Δt)
            ws_1 = compute_w_plus(d_grid[0:], D_grid[0:], True, Δt)

            curr_bs[0] = True
            curr_ws.w[0] = ws_1[0]

            curr_bs[1:] = ws_1[1:] >= ws_0
            curr_ws.w[1:] = np.where(curr_bs[1:], ws_1[1:], ws_0)

            # save results
            ts.append(t)
            bs.append(curr_bs.copy())

            # prepare next round
            curr_ws, next_ws = next_ws, curr_ws

            Δt = Δt_step.compute_Δt(dsp.Δd)

        # prepare forward propagation
        ts.reverse()
        bs.reverse()

        self.__ts: Final = ts
        self.__bs: Final = bs
        self.__dsp: Final = dsp
        self.__stable: Final = Δt_step.stable

        self.__gp: Final = gp
        self.__model: Final = model

    @property
    def ts(self) -> Sequence[float]:
        return self.__ts

    @property
    def stable(self) -> bool:
        return self.__stable

    def b(self, kt: int, D: float) -> bool:
        return self.__bs[kt][self.__dsp.D_to_k(D, clamp=True)]

    def bs(self) -> npt.NDArray[np.bool]:
        return np.array(self.__bs, dtype=np.bool)

    def Ds(self) -> npt.NDArray[np.float32]:
        return np.array([self.__dsp.k_to_D(k) for k in range(self.__gp.D_int + 2)])

    def grid(self) -> GridParams:
        return self.__gp

    def model(self) -> Model:
        return self.__model


# ==================================================================================================
# Simulation


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class ModelParamsSpace:
    α: npt.NDArray[np.float32]
    μ: npt.NDArray[np.float32]
    r: npt.NDArray[np.float32]
    k: npt.NDArray[np.int32]


@final
@dataclasses.dataclass(kw_only=True, slots=True, frozen=True)
class ModelParams:
    α: float
    μ: float
    r: float
    k: int

    def to_segment_space(self) -> ModelParams:
        return ModelParams(
            α=self.α,
            μ=self.μ / self.k,
            r=self.r / self.k,
            k=1,
        )


@final
@dataclasses.dataclass(kw_only=True, slots=True, frozen=True)
class InitialParams:
    V0: float
    D0: float

    def scale_time(self, k: float) -> InitialParams:
        return InitialParams(
            V0=self.V0 / k,
            D0=self.D0 / k,
        )

    def to_segment_space(self, k: float) -> InitialParams:
        return self.scale_time(1 / k)

    def to_integral_space(self, k: float) -> InitialParams:
        return self.scale_time(k)


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class SingleCurve:
    b_opt: npt.NDArray[np.bool]
    V_opt: npt.NDArray[np.float32]
    D_opt: npt.NDArray[np.float32]

    def scale_time(self, k: float) -> SingleCurve:
        return SingleCurve(
            b_opt=np.array(self.b_opt),
            V_opt=(self.V_opt / k).astype(np.float32, copy=False),
            D_opt=(self.D_opt / k).astype(np.float32, copy=False),
        )

    def to_segment_space(self, k: float) -> SingleCurve:
        return self.scale_time(1 / k)

    def to_integral_space(self, k: float) -> SingleCurve:
        return self.scale_time(k)


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class SimulationCurvesPack:
    ts: npt.NDArray[np.float32]
    Ds: npt.NDArray[np.float32]
    bs: npt.NDArray[np.bool]
    curves: list[tuple[InitialParams, SingleCurve]]

    def scale_time(self, k: float) -> SimulationCurvesPack:
        return SimulationCurvesPack(
            ts=self.ts / k,
            Ds=self.Ds / k,
            bs=np.array(self.bs),
            curves=[(key.scale_time(k), curve.scale_time(k)) for key, curve in self.curves],
        )

    def to_segment_space(self, k: float) -> SimulationCurvesPack:
        return self.scale_time(1 / k)

    def to_integral_space(self, k: float) -> SimulationCurvesPack:
        return self.scale_time(k)


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class SimulationCaseResults:
    segments: SimulationCurvesPack
    integral: SimulationCurvesPack


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class SimulationBundle:
    results: dict[ModelParams, SimulationCaseResults]


def simulate(controller: Controller, ips: Sequence[InitialParams], k: int) -> SimulationCurvesPack:
    assert k > 0

    md: Final = controller.model()
    ts: Final = controller.ts
    assert len(ts) > 0

    t0: Final = ts[0]
    t1: Final = ts[-1]
    ΔT: Final = t1 - t0

    def compute_curve(ip: InitialParams) -> SingleCurve:
        all_b_opt: Final = list[npt.NDArray[np.bool]]()
        all_V_opt: Final = list[npt.NDArray[np.float32]]()
        all_D_opt: Final = list[npt.NDArray[np.float32]]()

        for _ in range(k):
            b_opt = np.zeros(len(ts), dtype=np.bool)
            V_opt = np.zeros(len(ts), dtype=np.float32)
            D_opt = np.zeros(len(ts), dtype=np.float32)
            V_opt[0] = ip.V0
            D_opt[0] = ip.D0

            for kt in range(len(ts)):
                V = V_opt[kt]
                D = D_opt[kt]

                b = controller.b(kt, D)
                b_opt[kt] = b

                if kt + 1 < len(ts):
                    Δt = ts[kt + 1] - ts[kt]
                    V_opt[kt + 1] = V + md.dotV(D, b) * Δt
                    D_opt[kt + 1] = max(D + md.dotD(D, b) * Δt, 0.0)

            assert len(V_opt) > 0
            assert len(D_opt) > 0
            ip = InitialParams(V0=V_opt[-1], D0=D_opt[-1])

            all_b_opt.append(b_opt)
            all_V_opt.append(V_opt)
            all_D_opt.append(D_opt)

        # Glue: one cell per time (no duplicate at segment boundaries)
        b_glued = np.concatenate([all_b_opt[0]] + [all_b_opt[i][1:] for i in range(1, k)])
        V_glued = np.concatenate([all_V_opt[0]] + [all_V_opt[i][1:] for i in range(1, k)])
        D_glued = np.concatenate([all_D_opt[0]] + [all_D_opt[i][1:] for i in range(1, k)])
        return SingleCurve(b_opt=b_glued, V_opt=V_glued, D_opt=D_glued)

    # Glued time axis and control surface (one cell per time)
    ts_arr: Final = np.array(ts, dtype=np.float32)
    bs_arr: Final = np.array(controller.bs(), dtype=np.bool)

    return SimulationCurvesPack(
        ts=np.concatenate([ts_arr] + [ts_arr[1:] + i * ΔT for i in range(1, k)], dtype=np.float32),
        Ds=controller.Ds(),
        bs=np.concatenate([bs_arr] + [bs_arr[1:]] * (k - 1), axis=0, dtype=np.bool),
        curves=[(ip, compute_curve(ip)) for ip in ips],
    )


@final
@dataclasses.dataclass(kw_only=True, slots=True)
class SimulationPack:
    params: ModelParams

    model_factory: Callable[[ModelParams], Model]
    space: ModelParamsSpace
    ips: Sequence[InitialParams]
    gp: GridParams


def simulate_cell(pack: SimulationPack) -> tuple[ModelParams, SimulationCaseResults]:
    def simulate_segments(params: ModelParams, k: int) -> SimulationCurvesPack:
        model: Final = pack.model_factory(params)
        controller: Final = Controller(model, pack.gp.to_segment_space(k))
        if not controller.stable:
            print(f"WARNING: unstable solution for: {params}\n", end="")

        ips: Final = [x.to_segment_space(k) for x in pack.ips]
        return simulate(controller, ips, k).to_integral_space(k)

    segments: Final = simulate_segments(pack.params.to_segment_space(), pack.params.k)
    integral: Final = simulate_segments(pack.params, 1)

    # Ensure that simulated trajectories start from the same points
    assert len(segments.curves) == len(integral.curves)
    for (s_ip, s_curve), (i_ip, i_curve) in zip(segments.curves, integral.curves):
        assert abs(s_ip.V0 - i_ip.V0) < 1e-6
        assert abs(s_ip.D0 - i_ip.D0) < 1e-6
        assert abs(s_curve.V_opt[0] - i_curve.V_opt[0]) < 1e-6
        assert abs(s_curve.D_opt[0] - i_curve.D_opt[0]) < 1e-6

    return (pack.params, SimulationCaseResults(segments=segments, integral=integral))


def simulate_space(
    model_factory: Callable[[ModelParams], Model],
    space: ModelParamsSpace,
    ips: Sequence[InitialParams],
    gp: GridParams,
) -> SimulationBundle:
    packs: Final = [
        SimulationPack(
            params=ModelParams(α=α, μ=μ, r=r, k=k),
            model_factory=model_factory,
            space=space,
            ips=ips,
            gp=gp,
        )
        for k, α, μ, r in itertools.product(space.k, space.α, space.μ, space.r)
    ]

    with concurrent.futures.ProcessPoolExecutor(mp_context=multiprocessing.get_context("fork")) as pool:
        results_dict = dict(pool.map(simulate_cell, packs, chunksize=10))
    return SimulationBundle(results=results_dict)


# ==================================================================================================


@final
class ModelExp(Model):
    __slots__: Final = "__mp"

    def __init__(self, mp: ModelParams) -> None:
        self.__mp: Final = mp

    @override
    def dotV(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
        mp: Final = self.__mp
        return int(b) * np.exp(-mp.μ * D)

    @override
    def dotD(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
        mp: Final = self.__mp
        return (((mp.α + 1) * int(b) - 1) * np.exp(-mp.μ * D) + mp.r * D).astype(np.float32, copy=False)


@final
class ModelHyper(Model):
    __slots__: Final = "__mp"

    def __init__(self, mp: ModelParams) -> None:
        self.__mp: Final = mp

    @override
    def dotV(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
        mp: Final = self.__mp
        return (int(b) / (1 + mp.μ * D)).astype(np.float32, copy=False)

    @override
    def dotD(self, D: npt.NDArray[np.float32], b: bool) -> npt.NDArray[np.float32]:
        mp: Final = self.__mp
        return (((mp.α + 1) * int(b) - 1) / (1 + mp.μ * D) + mp.r * D).astype(np.float32, copy=False)

In [ ]:
BASE = 10

packs = ModelParamsSpace(
    α=np.linspace(0, 2.00, num=BASE, dtype=np.float32),
    μ=np.linspace(0, 10.0, num=BASE, dtype=np.float32),
    r=np.linspace(0, 1.00, num=BASE, dtype=np.float32),
    k=np.array([2, 3, 4, 5, 6], dtype=np.int32),
)
initial_params = [InitialParams(V0=0.0, D0=D0) for D0 in np.linspace(0, 0.4, 200)]
grid_params = GridParams(
    D_min=0,
    D_max=1.5,
    D_int=200,
    T0=0,
    T1=1,
    CFL=0.9,
    Δt_min=0.0001,
    Δt_max=0.1000,
)

bundle = simulate_space(ModelExp, packs, initial_params, grid_params)


def _all_curves(results: SimulationCaseResults):
    for pack in (results.segments, results.integral):
        for _, curve in pack.curves:
            yield curve


curves_all = [c for x in bundle.results.values() for c in _all_curves(x)]
print(max(max(c.D_opt) for c in curves_all))
print(max(max(c.V_opt) for c in curves_all))

In [ ]:
import plotly.colors
import plotly.graph_objects as go
import plotly.subplots

key, data = list(bundle.results.items())[0 * BASE**3 + np.sum(int(BASE * 0.6) * BASE ** np.array([2, 1, 0]))]
assert isinstance(key, ModelParams)
assert isinstance(data, SimulationCaseResults)

# Create figure with subplots
fig: go.Figure = plotly.subplots.make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        f"Optimal control b<sub>p</sub><sup>q</sup>, α={key.α:.2f}, μ={key.μ:.2f}, r={key.r:.2f} (k={key.k})",
        f"Phase portrait, α={key.α:.2f}, μ={key.μ:.2f}, r={key.r:.2f}",
    ),
    horizontal_spacing=0.12,
    shared_xaxes=True,
)

# Left subplot: D-T trajectories with contour (integral control surface)
fig.add_trace(
    go.Contour(
        z=data.segments.bs.astype(np.int32),
        x=data.segments.Ds,
        y=data.segments.ts,
        colorscale=[[0, "gray"], [1, "rgb(70, 230, 30)"]],
        showscale=True,
    ),
    row=1,
    col=1,
)

# Trajectories: from local segments (red) and from one integral run (blue)
for _, curve in data.segments.curves:
    fig.add_trace(
        go.Scatter(
            x=curve.D_opt,
            y=data.segments.ts,
            mode="lines",
            line=dict(color="red", width=2),
            name="from segments",
            legendgroup="segments",
            showlegend=False,
        ),
        row=1,
        col=1,
    )
for _, curve in data.integral.curves:
    fig.add_trace(
        go.Scatter(
            x=curve.D_opt,
            y=data.integral.ts,
            mode="lines",
            line=dict(color="blue", width=2),
            name="from integral",
            legendgroup="integral",
            showlegend=False,
        ),
        row=1,
        col=1,
    )

# Right subplot: Phase portraits (both curves)
for _, curve in data.segments.curves:
    fig.add_trace(
        go.Scatter(
            x=curve.D_opt,
            y=curve.V_opt,
            mode="lines",
            line=dict(color="red", width=2),
            name="from segments",
            legendgroup="segments",
            showlegend=False,
        ),
        row=1,
        col=2,
    )
for _, curve in data.integral.curves:
    fig.add_trace(
        go.Scatter(
            x=curve.D_opt,
            y=curve.V_opt,
            mode="lines",
            line=dict(color="blue", width=2),
            name="from integral",
            legendgroup="integral",
            showlegend=False,
        ),
        row=1,
        col=2,
    )

# Update axes labels
fig.update_xaxes(title_text="D", row=1, col=1)
fig.update_yaxes(title_text="t", row=1, col=1)
fig.update_xaxes(title_text="D", row=1, col=2)
fig.update_yaxes(title_text="V", row=1, col=2)

fig.update_xaxes(range=[data.integral.Ds.min(), data.integral.Ds.max()], row=1, col=1)
fig.update_yaxes(range=[data.integral.ts.min(), data.integral.ts.max()], row=1, col=1)
fig.update_xaxes(range=[data.integral.Ds.min(), data.integral.Ds.max()], row=1, col=2)

# Update layout
fig.update_layout(
    width=1200,
    height=600,
    showlegend=True,
    margin=dict(l=50, r=50, t=80, b=50),
)
fig.show()

In [ ]:
# Create figure with subplots
fig: go.Figure = plotly.subplots.make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        f"CDF V<sub>segments</sub> / V<sub>integral</sub>",
        f"CDF D<sub>segments</sub> - D<sub>integral</sub>",
    ),
    horizontal_spacing=0.12,
    shared_xaxes=True,
)

# Left: V degradation
for i, k in enumerate(np.sort(np.unique([x.k for x in bundle.results.keys()]))):
    rates = [
        segments.V_opt[-1] / integral.V_opt[-1]
        for key, case in bundle.results.items()
        for (_, segments), (_, integral) in zip(case.segments.curves, case.integral.curves)
        if key.k == k
        if segments.b_opt.min() < 0.5 or integral.b_opt.min() < 0.5  # if we have any control over the process
    ]
    data = np.histogram(rates, bins=1000, density=True)

    fig.add_trace(
        go.Scatter(
            x=data[1],
            y=np.cumsum(data[0]) / np.sum(data[0]),
            mode="lines",
            name=f"k={k}",
            line_shape="hvh",
            marker_color=plotly.colors.DEFAULT_PLOTLY_COLORS[i],
        ),
        row=1,
        col=1,
    )

# Left: D degradation
for i, k in enumerate(np.sort(np.unique([x.k for x in bundle.results.keys()]))):
    rates = [
        segments.D_opt[-1] - integral.D_opt[-1]
        for key, case in bundle.results.items()
        for (_, segments), (_, integral) in zip(case.segments.curves, case.integral.curves)
        if key.k == k
        # if segments.b_opt.min() < 0.5 or integral.b_opt.min() < 0.5  # if we have any control over the process
    ]
    data = np.histogram(rates, bins=1000, density=True)

    fig.add_trace(
        go.Scatter(
            x=data[1],
            y=np.cumsum(data[0]) / np.sum(data[0]),
            mode="lines",
            name=f"k={k}",
            line_shape="hvh",
            marker_color=plotly.colors.DEFAULT_PLOTLY_COLORS[i],
        ),
        row=1,
        col=2,
    )

# Update axes labels
fig.update_xaxes(title_text="V(segments/integral)", row=1, col=1)
fig.update_yaxes(title_text="", row=1, col=1)
fig.update_xaxes(title_text="D(segments - integral)", row=1, col=2)
fig.update_yaxes(title_text="", row=1, col=2)

# Update layout
fig.update_layout(
    width=1200,
    height=600,
    showlegend=True,
    margin=dict(l=50, r=50, t=80, b=50),
)

names = set()
fig.for_each_trace(lambda trace: trace.update(showlegend=False) if (trace.name in names) else names.add(trace.name))

fig.show()

In [ ]:
import random

fig = go.Figure()

for key, case in bundle.results.items():
    if key.k != k:
        continue

    for (_, segments), (_, integral) in zip(case.segments.curves, case.integral.curves):
        if not (segments.b_opt.min() < 0.5 or integral.b_opt.min()):
            continue
        if not (segments.V_opt[-1] - integral.V_opt[-1] > 0.01):
            continue
        if random.random() > 0.1:
            continue

        for name, color, pack, curve in [
            ("segments", "red", case.segments, segments),
            ("integral", "blue", case.integral, integral),
        ]:
            fig.add_trace(
                go.Scatter(
                    x=curve.D_opt,
                    y=curve.V_opt,
                    mode="lines",
                    line=dict(color=color, width=2),
                    name=f"from {name}",
                    legendgroup=name,
                    showlegend=False,
                ),
            )

# Update layout
fig.update_layout(
    width=800,
    height=800,
    showlegend=True,
    margin=dict(l=50, r=50, t=80, b=50),
)

fig.show()